# Notebook 04 — GenAI Tracing

**RHOAI 3.5 | MLflow Tracing API | GenAI Workflow Mode**

MLflow 3.x ships a first-class **Tracing API** for LLM/GenAI applications. RHOAI 3.5 exposes a dedicated **GenAI workflow mode** in the dashboard with:
- **Overview** tab — trace timeline and summary stats
- **Traces** tab — searchable list of individual traces
- **Sessions** tab — multi-turn conversation grouping

This notebook demonstrates:
1. Tracing with `@mlflow.trace` decorator
2. Manual span creation for fine-grained control
3. Session tracking (multi-turn conversations)
4. Simulated LLM pipeline tracing
5. Trace Archival — the RHOAI 3.5 feature that moves span payloads from PostgreSQL to S3

---
**No external LLM API key required** — this notebook simulates LLM calls with deterministic mock functions.

In [ ]:
import mlflow
import mlflow.tracking
from mlflow.entities import SpanType
import time, random, json, uuid
from typing import Optional

EXPERIMENT_NAME = "rhoai-genai-tracing-demo"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Active experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI     : {mlflow.get_tracking_uri()}")
print()
print("RHOAI Dashboard workflow mode: set to 'GenAI' to see Traces and Sessions tabs.")
print("  Develop & train > Experiments (MLflow) > [experiment] > toggle workflow type")

## Simulated LLM Components

These mock functions stand in for real LLM/retrieval calls so the notebook runs without API keys.
Replace them with your actual LLM SDK calls in production.

In [ ]:
MOCK_KNOWLEDGE_BASE = [
    {"id": "doc-001", "content": "RHOAI 3.5 ships MLflow 3.13.0 with the Kubernetes auth plugin."},
    {"id": "doc-002", "content": "MLflow Trace Archival moves span payloads from PostgreSQL to S3 via a CronJob."},
    {"id": "doc-003", "content": "The RHOAI dashboard GenAI workflow mode shows Traces, Sessions, and Overview tabs."},
    {"id": "doc-004", "content": "Every MLflow API call in RHOAI is authorized by Kubernetes RBAC via SelfSubjectAccessReview."},
    {"id": "doc-005", "content": "The minimum MLflow SDK version for RHOAI 3.5 is mlflow[kubernetes]>=3.11."},
]

def mock_embed(text: str) -> list:
    """Simulate embedding generation (deterministic)."""
    time.sleep(0.02)
    random.seed(hash(text) % (2**32))
    return [round(random.gauss(0, 1), 4) for _ in range(8)]

def mock_retrieve(query_embedding: list, top_k: int = 2) -> list:
    """Simulate vector DB retrieval."""
    time.sleep(0.03)
    return random.sample(MOCK_KNOWLEDGE_BASE, min(top_k, len(MOCK_KNOWLEDGE_BASE)))

def mock_llm_generate(prompt: str, temperature: float = 0.7, max_tokens: int = 256) -> dict:
    """Simulate an LLM completion call."""
    time.sleep(0.05)
    token_count = len(prompt.split()) + 20
    return {
        "model"       : "rhoai-llm-mock-v1",
        "content"     : f"Based on the context, here is a concise answer to: '{prompt[:60]}...' [mocked response]",
        "usage"       : {"prompt_tokens": len(prompt.split()), "completion_tokens": 20, "total_tokens": token_count},
        "finish_reason": "stop",
    }

print("Mock LLM components ready.")

## Section 1 — Tracing with `@mlflow.trace` Decorator

The simplest way to trace a function. MLflow automatically captures:
- Function inputs and outputs
- Execution duration
- Exceptions (if any)

Span types from `mlflow.entities.SpanType`: `LLM`, `RETRIEVER`, `EMBEDDING`, `TOOL`, `CHAIN`, `AGENT`, `RERANKER`, `UNKNOWN`

In [ ]:
@mlflow.trace(span_type=SpanType.EMBEDDING, name="embed-query")
def embed_query(text: str) -> list:
    return mock_embed(text)

@mlflow.trace(span_type=SpanType.RETRIEVER, name="retrieve-docs")
def retrieve_docs(query_embedding: list, top_k: int = 2) -> list:
    docs = mock_retrieve(query_embedding, top_k=top_k)
    # Attach span attributes for dashboard display
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("retriever.top_k", top_k)
        span.set_attribute("retriever.doc_ids", [d["id"] for d in docs])
    return docs

@mlflow.trace(span_type=SpanType.LLM, name="llm-generate")
def llm_generate(prompt: str, temperature: float = 0.7, max_tokens: int = 256) -> dict:
    response = mock_llm_generate(prompt, temperature=temperature, max_tokens=max_tokens)
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("llm.model",              response["model"])
        span.set_attribute("llm.temperature",        temperature)
        span.set_attribute("llm.max_tokens",         max_tokens)
        span.set_attribute("llm.usage.total_tokens", response["usage"]["total_tokens"])
        span.set_attribute("llm.finish_reason",      response["finish_reason"])
    return response

@mlflow.trace(span_type=SpanType.CHAIN, name="rag-pipeline")
def rag_pipeline(user_query: str, top_k: int = 2, temperature: float = 0.7) -> str:
    """Retrieval-Augmented Generation pipeline — full trace hierarchy."""
    # Step 1: Embed the query
    query_embedding = embed_query(user_query)

    # Step 2: Retrieve relevant documents
    docs = retrieve_docs(query_embedding, top_k=top_k)

    # Step 3: Build prompt with context
    context = "\n".join([f"- {d['content']}" for d in docs])
    prompt = f"Context:\n{context}\n\nQuestion: {user_query}\nAnswer:"

    # Step 4: Generate answer
    response = llm_generate(prompt, temperature=temperature)

    return response["content"]


# Run the pipeline — this creates a trace in MLflow
with mlflow.start_run(run_name="rag-trace-demo"):
    mlflow.set_tags({"notebook": "04_genai_tracing", "workflow": "RAG"})

    answer = rag_pipeline(
        user_query="What version of MLflow is in RHOAI 3.5 and how does authentication work?",
        top_k=2,
        temperature=0.5,
    )
    mlflow.log_param("pipeline_type", "RAG")

print(f"Answer: {answer}")
print()
print("View the trace in the RHOAI Dashboard:")
print("  Experiments (MLflow) > rhoai-genai-tracing-demo > [run] > Traces tab")

## Section 2 — Manual Span Creation

Use `mlflow.start_span()` for precise control: add spans in non-function contexts,
or when you need to capture intermediate state mid-function.

In [ ]:
def agent_with_tools(user_request: str) -> str:
    """Simulated agent that decides which tool to call."""
    with mlflow.start_span(name="agent-decision", span_type=SpanType.AGENT) as agent_span:
        agent_span.set_inputs({"user_request": user_request})
        agent_span.set_attribute("agent.model", "rhoai-agent-mock")

        # Span: tool routing decision
        with mlflow.start_span(name="route-request", span_type=SpanType.TOOL) as route_span:
            time.sleep(0.02)
            tool = "knowledge_base" if "RHOAI" in user_request else "calculator"
            route_span.set_inputs({"query": user_request})
            route_span.set_outputs({"selected_tool": tool})

        # Span: tool execution
        with mlflow.start_span(name=f"execute-{tool}", span_type=SpanType.TOOL) as tool_span:
            if tool == "knowledge_base":
                emb  = mock_embed(user_request)
                docs = mock_retrieve(emb, top_k=1)
                result = docs[0]["content"] if docs else "No results"
            else:
                result = str(eval("2 + 2"))  # noqa: S307 — safe here, demo only

            tool_span.set_inputs({"query": user_request, "tool": tool})
            tool_span.set_outputs({"result": result})
            tool_span.set_attribute("tool.name", tool)

        # Span: final response generation
        with mlflow.start_span(name="generate-response", span_type=SpanType.LLM) as llm_span:
            response = mock_llm_generate(f"User: {user_request}\nContext: {result}")
            llm_span.set_inputs({"prompt_preview": user_request[:100]})
            llm_span.set_outputs({"response": response["content"]})
            llm_span.set_attribute("llm.total_tokens", response["usage"]["total_tokens"])

        agent_span.set_outputs({"final_response": response["content"]})
        return response["content"]


with mlflow.start_run(run_name="agent-manual-spans"):
    mlflow.set_tags({"notebook": "04_genai_tracing", "workflow": "Agent"})

    result1 = agent_with_tools("What is the minimum MLflow SDK version for RHOAI 3.5?")
    result2 = agent_with_tools("Calculate the answer")

    print(f"Result 1: {result1[:80]}...")
    print(f"Result 2: {result2[:80]}...")

## Section 3 — Session Tracking (Multi-Turn Conversations)

Sessions group multiple traces from the same user conversation.
The RHOAI Dashboard **Sessions** tab aggregates traces that share the same `mlflow.trace.session_id` attribute.

Set the session ID on the root span with `span.set_attribute("mlflow.trace.session_id", session_id)` inside a `mlflow.start_span` context.

> **Note:** `mlflow.set_trace_tags()` was not available in MLflow 3.13 (RHOAI 3.5).
> Use `span.set_attribute()` on the root span instead — the Sessions tab reads the same attribute.

In [ ]:
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}")
print()

conversation = [
    "What is RHOAI?",
    "How does MLflow authentication work in RHOAI 3.5?",
    "What is trace archival and when should I enable it?",
]

with mlflow.start_run(run_name="multi-turn-session"):
    mlflow.set_tags({
        "notebook"  : "04_genai_tracing",
        "workflow"  : "ChatBot",
        "session_id": session_id,
    })

    for turn, user_message in enumerate(conversation, 1):
        # start_span creates the trace; set_attribute stamps session_id on the root span
        # so the RHOAI Sessions tab can group these traces together
        with mlflow.start_span(name=f"turn-{turn}", span_type=SpanType.CHAIN) as span:
            span.set_inputs({"message": user_message, "turn": turn})
            span.set_attribute("mlflow.trace.session_id", session_id)

            emb     = embed_query(user_message)
            docs    = retrieve_docs(emb, top_k=2)
            context = " ".join([d["content"] for d in docs])
            resp    = llm_generate(f"{context}\nQ: {user_message}", temperature=0.6)
            answer  = resp["content"]

            span.set_outputs({"response": answer})

        mlflow.log_metric(f"turn_{turn}_tokens", len(user_message.split()) + 20, step=turn)
        print(f"  Turn {turn}: {user_message[:50]}...")
        print(f"          → {answer[:70]}...")

print()
print("Session traces logged. View in Dashboard > Sessions tab.")
print(f"  Filter by session_id: {session_id}")

## Section 4 — Search and Retrieve Traces

Use `mlflow.search_traces()` to query traces programmatically.
The RHOAI Dashboard search bar supports the same filter syntax.

**Important for RHOAI 3.5:** After trace archival, filters on `trace.text`, `span.content`, and `span.attributes.*` will NOT match archived spans. Metadata-based filters (tags, timestamps, status) still work.

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    print("Experiment not found — run the cells above first.")
else:
    exp_id = experiment.experiment_id

    # Search all traces in the experiment
    traces = mlflow.search_traces(
        experiment_ids=[exp_id],
        order_by=["timestamp_ms DESC"],
        max_results=20,
    )

    print(f"Found {len(traces)} traces in '{EXPERIMENT_NAME}'.")
    if len(traces) > 0:
        cols = ["trace_id", "request_id", "status", "execution_time_ms", "timestamp_ms"]
        available = [c for c in cols if c in traces.columns]
        print(traces[available].head(10).to_string(index=False))

In [ ]:
# Retrieve a specific trace and inspect its spans
client = mlflow.tracking.MlflowClient()

if len(traces) > 0:
    first_trace_id = traces.iloc[0]["trace_id"] if "trace_id" in traces.columns else None
    if first_trace_id:
        trace = client.get_trace(trace_id=first_trace_id)
        print(f"Trace ID    : {trace.info.request_id}")
        print(f"Status      : {trace.info.status}")
        print(f"Duration    : {trace.info.execution_time_ms} ms")
        print(f"Tags        : {trace.info.tags}")
        print()
        if trace.data and trace.data.spans:
            print(f"Spans ({len(trace.data.spans)}):")
            for span in trace.data.spans:
                indent = "  " if span.parent_id else ""
                print(f"  {indent}[{span.span_type}] {span.name} — {span.status}")
    else:
        print("Could not extract trace ID from result.")
else:
    print("No traces found.")

## Section 5 — Trace Archival (RHOAI 3.5 Feature)

**Trace Archival** is a new RHOAI 3.5 feature that moves trace span payloads from PostgreSQL to S3-compatible object storage, reducing database size for long-running systems.

### How it works
1. A Kubernetes CronJob runs on the configured `schedule`
2. Spans older than `retention` (e.g. `30d`) are serialised as OTLP protobuf
3. Files are stored at: `{archive_location}/{experiment_id}/traces/{trace_id}/artifacts/traces.pb`
4. PostgreSQL rows are updated with tag `mlflow.trace.spansLocation=archived`
5. Archived traces remain viewable in the dashboard (with higher latency)

### Configuration (Admin task — OpenShift CLI)

Enable archival by patching the `MLflowTrackingServer` CR:

In [ ]:
# This cell shows the admin configuration — do not run it from the notebook.
# Run these oc commands in your cluster terminal.

ARCHIVAL_CONFIG_EXAMPLE = """
# Patch your MLflowTrackingServer CR to enable trace archival:

oc patch mlflowtrackingserver mlflow -n <your-namespace> --type=merge -p '
spec:
  traceArchival:
    enabled: true
    schedule: "0 2 * * *"           # Run at 2 AM daily
    retention: 30d                   # Archive spans older than 30 days
    maxTracesPerPass: 1000           # Batch size per CronJob run (default: 1000)
    location:
      bucket: my-s3-bucket
      prefix: mlflow-trace-archive
    longRetentionAllowlist:          # Experiments excluded from archival
      - production-rag-eval
'

# Verify the CronJob was created:
oc get cronjob -n <your-namespace>

# Trigger archival immediately (for testing):
oc create job --from=cronjob/mlflow-trace-archival manual-archive-$(date +%s) -n <your-namespace>
"""

print(ARCHIVAL_CONFIG_EXAMPLE)

In [ ]:
# Per-experiment retention can be set from the SDK or CLI
# This overrides the server-level default for a specific experiment

RETENTION_EXAMPLE = """
# Set per-experiment retention from Python:
client = mlflow.tracking.MlflowClient()
client.set_experiment_tag(
    experiment_id="<exp_id>",
    key="mlflow.traceArchival.retention",
    value="90d",   # Keep traces in DB for 90 days before archiving
)

# Or archive a specific trace immediately from the dashboard:
# Experiments > [experiment] > Traces > [trace] > Archive now

# Or via CLI:
mlflow traces archive --experiment-id <exp_id> --trace-id <trace_id>
"""

print(RETENTION_EXAMPLE)

### Archival and Search Behaviour

Once a trace is archived, these filters **still work**:
- By timestamp, status, duration
- By tag values
- By metadata columns (experiment, run)

These filters **do NOT work** on archived spans:
- `trace.text` (full-text search on span content)
- `span.content`
- `span.attributes.*`

Use `longRetentionAllowlist` to exempt high-value experiments from archival.

In [ ]:
# Check archival status of a trace via its tags
if len(traces) > 0 and "trace_id" in traces.columns:
    first_trace_id = traces.iloc[0]["trace_id"]
    trace = client.get_trace(trace_id=first_trace_id)

    archival_tags = {
        k: v for k, v in (trace.info.tags or {}).items()
        if k.startswith("mlflow.trace")
    }

    if archival_tags:
        print("Archival-related tags on this trace:")
        for k, v in archival_tags.items():
            print(f"  {k}: {v}")
    else:
        print("No archival tags found — trace has not been archived yet.")
        print()
        print("After archival, the following tags would appear:")
        print("  mlflow.trace.spansLocation  : 'archived'")
        print("  mlflow.trace.archiveLocation: 's3://bucket/prefix/exp_id/traces/trace_id/'")
        print("  mlflow.trace.archivalFailure: 'error message' (only if archival failed)")
else:
    print("No traces to check.")

## RHOAI Dashboard: GenAI Workflow Mode

To view these traces in the RHOAI Dashboard GenAI view:

1. Go to `Develop & train > Experiments (MLflow)`
2. Select the `rhoai-genai-tracing-demo` experiment
3. Click the **workflow type toggle** in the top-right → select **GenAI**
4. You now see three tabs:
   - **Overview** — trace count, latency distribution, error rate timeline
   - **Traces** — searchable table of all traces with duration, status, span count
   - **Sessions** — grouped by `mlflow.trace.session_id` tag
5. Click any trace to see the span waterfall diagram
6. Use the search bar with filters like: `status = 'OK'`, `execution_time_ms > 100`

## Summary

| Feature | API / Config |
|---------|-------------|
| Decorator tracing | `@mlflow.trace(span_type=SpanType.LLM)` |
| Manual spans | `mlflow.start_span(name, span_type)` |
| Span attributes | `span.set_attribute(key, value)` |
| Session tracking | `mlflow.set_trace_tags({"mlflow.trace.session_id": ...})` |
| Search traces | `mlflow.search_traces(experiment_ids, filter_string)` |
| Retrieve trace | `client.get_trace(trace_id)` |
| Trace archival | `spec.traceArchival.enabled: true` in MLflow CR (admin) |
| Per-experiment retention | `client.set_experiment_tag("mlflow.traceArchival.retention", "90d")` |

**Next:** `05_model_registry.ipynb` — register, version, and serve models from the MLflow registry.